# CauNagi downstream analysis tutorial

This notebook is a non-executed template for analysing outputs from a completed CauNagi iteration. It does not rerun model training. Update the paths and set the guarded flags to `True` only when you are ready to create new downstream results.

In [ ]:
from pathlib import Path
import ast
import json
import pickle
import sys

import anndata as ad
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy.sparse import issparse

# Resolve the repository root from either the repository directory or tutorials/.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), Path.cwd().parent) if (path / "Main_code").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the repository root containing Main_code/")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Use the repository-local path first, then the original project-level path.
TEMP_PATH = next(
    (
        path
        for path in (PROJECT_ROOT / "temp_path", PROJECT_ROOT.parent / "temp_path")
        if path.is_dir()
    ),
    PROJECT_ROOT / "temp_path",
)

FINAL_ITERATION = 0
ITERATION_DIR = TEMP_PATH / str(FINAL_ITERATION)
STAGEDATA_DIR = ITERATION_DIR / "stagedata"
DATASET_PATH = STAGEDATA_DIR / "dataset.h5ad"
ATTRIBUTE_PATH = STAGEDATA_DIR / "attribute.pkl"
IDREM_RESULTS_DIR = ITERATION_DIR / "idremResults"

MARKER_RESULTS_DIR = PROJECT_ROOT / "results" / "caunagi_markers"
DRIVER_RESULTS_DIR = PROJECT_ROOT / "results" / "driver_genes"
CELLTYPE_DRIVER_DIR = PROJECT_ROOT / "results" / "driver_genes" / "per_celltype"
CASCADE_RESULTS_DIR = PROJECT_ROOT / "results" / "cascade_classification"
DOWNSTREAM_RESULTS_DIR = PROJECT_ROOT / "results" / "downstream_tutorial"

print(f"Project root: {PROJECT_ROOT}")
print(f"CauNagi output root: {TEMP_PATH}")
print(f"Final iteration: {FINAL_ITERATION}")

## 1. Validate the completed CauNagi output

A completed iteration should contain staged AnnData files, `dataset.h5ad`, `attribute.pkl`, `edges.txt`, and an `idremResults/` directory.

In [ ]:
required_paths = {
    "stagedata directory": STAGEDATA_DIR,
    "merged dataset": DATASET_PATH,
    "attribute file": ATTRIBUTE_PATH,
    "iDREM results": IDREM_RESULTS_DIR,
}

for label, path in required_paths.items():
    print(f"{label}: {'OK' if path.exists() else 'MISSING'} -> {path}")

if not DATASET_PATH.is_file():
    raise FileNotFoundError(f"Missing completed dataset: {DATASET_PATH}")

stage_files = sorted(
    STAGEDATA_DIR.glob("*.h5ad"),
    key=lambda path: int(path.stem) if path.stem.isdigit() else 10**9,
)
print("Stage files:", [path.name for path in stage_files])

## 2. Load the merged AnnData output

The merged dataset contains the stage labels, Leiden clusters, cell-type annotations, learned gene weights, and other attributes needed for downstream analysis.

In [ ]:
adata = sc.read_h5ad(DATASET_PATH)

if ATTRIBUTE_PATH.is_file():
    with ATTRIBUTE_PATH.open("rb") as handle:
        attributes = pickle.load(handle)
    if isinstance(attributes, dict):
        adata.uns.update(attributes)

print(adata)
print("Observation columns:", list(adata.obs.columns))
print("Layers:", list(adata.layers.keys()))
print("Embeddings:", list(adata.obsm.keys()))

## 3. Summarise disease states, cell types, and clusters

In [ ]:
def first_existing(columns, candidates):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None

stage_key = first_existing(adata.obs.columns, ["stage", "disease_stage"])
celltype_key = first_existing(adata.obs.columns, ["celltype", "CellType"])
cluster_key = first_existing(adata.obs.columns, ["leiden", "cluster"])

if stage_key is None or celltype_key is None:
    raise KeyError("The merged dataset must contain stage and cell-type annotations.")

summary = (
    adata.obs.groupby([stage_key, celltype_key], observed=False)
    .size()
    .rename("cell_count")
    .reset_index()
)
display(summary)

if cluster_key is not None:
    cluster_summary = (
        adata.obs.groupby([stage_key, cluster_key], observed=False)
        .size()
        .rename("cell_count")
        .reset_index()
    )
    display(cluster_summary.head(20))

fig, ax = plt.subplots(figsize=(10, 5))
sns.countplot(data=adata.obs, x=stage_key, hue=celltype_key, ax=ax)
ax.set_title("Cell-type composition across disease states")
ax.set_xlabel("Disease state or stage")
ax.set_ylabel("Number of cells")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 4. Inspect the learned gene weights

The `geneWeight` layer records iterative gene-level evidence. The following code calculates the mean weight per gene and, when cell types are available, a cell-type-specific summary.

In [ ]:
if "geneWeight" not in adata.layers:
    print("The geneWeight layer is not available in this dataset.")
else:
    gene_weight = adata.layers["geneWeight"]
    gene_weight = gene_weight.toarray() if issparse(gene_weight) else np.asarray(gene_weight)

    gene_weight_df = pd.DataFrame(
        {
            "gene": adata.var_names.astype(str),
            "mean_gene_weight": gene_weight.mean(axis=0),
            "max_gene_weight": gene_weight.max(axis=0),
        }
    ).sort_values("mean_gene_weight", ascending=False)

    display(gene_weight_df.head(30))
    gene_weight_df.to_csv(
        DOWNSTREAM_RESULTS_DIR / "gene_weight_summary.csv",
        index=False,
    )

    celltype_gene_weight = (
        pd.DataFrame(gene_weight, columns=adata.var_names.astype(str), index=adata.obs_names)
        .assign(cell_type=adata.obs[celltype_key].astype(str).to_numpy())
        .groupby("cell_type")
        .mean()
        .T
        .reset_index(names="gene")
    )
    display(celltype_gene_weight.head())
    celltype_gene_weight.to_csv(
        DOWNSTREAM_RESULTS_DIR / "celltype_gene_weight_summary.csv",
        index=False,
    )

## 5. Read the temporal graph

CauNagi stores temporal cluster connections in `edges.txt`. The file contains a Python-literal dictionary whose keys identify adjacent stage pairs.

In [ ]:
edges_path = ITERATION_DIR / "edges.txt"
if edges_path.is_file():
    edges = ast.literal_eval(edges_path.read_text(encoding="utf-8"))
    edge_rows = []
    graph = nx.DiGraph()

    for transition, transition_edges in edges.items():
        for source, target in transition_edges:
            edge_rows.append(
                {
                    "transition": transition,
                    "source": str(source),
                    "target": str(target),
                }
            )
            graph.add_edge(f"{transition}:{source}", f"{int(transition) + 1}:{target}")

    edge_df = pd.DataFrame(edge_rows)
    display(edge_df.head(20))
    DOWNSTREAM_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    edge_df.to_csv(DOWNSTREAM_RESULTS_DIR / "temporal_edges.csv", index=False)

    print(f"Temporal graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
else:
    print(f"Missing temporal graph: {edges_path}")

## 6. Inspect iDREM outputs and extract TF evidence

The iDREM result directory contains trajectory visualisations and `DREM.json` files. The project parser is reused here so TF scores are extracted using the same logic as the driver-gene analysis script.

In [ ]:
idrem_files = sorted(IDREM_RESULTS_DIR.rglob("*")) if IDREM_RESULTS_DIR.is_dir() else []
display(pd.DataFrame({"path": [str(path.relative_to(TEMP_PATH)) for path in idrem_files if path.is_file()]}).head(50))

from Candidate_Regulator_Screening.driver_gene_identification import robust_parse_drem_json

tf_rows = []
for drem_file in IDREM_RESULTS_DIR.rglob("DREM.json"):
    tf_scores = robust_parse_drem_json(str(drem_file))
    trajectory = drem_file.parent.name
    for gene, score in tf_scores.items():
        tf_rows.append(
            {
                "trajectory": trajectory,
                "tf": gene,
                "drem_score": score,
                "source_file": str(drem_file),
            }
        )

tf_evidence = (
    pd.DataFrame(tf_rows)
    .sort_values("drem_score", ascending=False)
    if tf_rows
    else pd.DataFrame(columns=["trajectory", "tf", "drem_score", "source_file"])
)
display(tf_evidence.head(30))
DOWNSTREAM_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
tf_evidence.to_csv(DOWNSTREAM_RESULTS_DIR / "idrem_tf_evidence.csv", index=False)

## 7. Load marker-analysis outputs

After `analyse_UNAGI()`, marker results are commonly stored as `hcmarkers.pkl`, `dynamic_markers.pkl`, and a progression-marker background file. The objects are kept in their native format and can be inspected before custom summarisation.

In [ ]:
def load_pickle_if_exists(path):
    if not path.is_file():
        print(f"Missing: {path}")
        return None
    with path.open("rb") as handle:
        value = pickle.load(handle)
    print(f"{path.name}: {type(value).__name__}")
    return value

hcmarkers = load_pickle_if_exists(MARKER_RESULTS_DIR / "hcmarkers.pkl")
dynamic_markers = load_pickle_if_exists(MARKER_RESULTS_DIR / "dynamic_markers.pkl")

if isinstance(hcmarkers, pd.DataFrame):
    display(hcmarkers.head())
elif isinstance(hcmarkers, dict):
    display(pd.Series(hcmarkers, name="hcmarker_value").head(20))

if isinstance(dynamic_markers, pd.DataFrame):
    display(dynamic_markers.head())
elif isinstance(dynamic_markers, dict):
    display(pd.Series(dynamic_markers, name="dynamic_marker_value").head(20))

## 8. Re-run marker analysis from a completed staged dataset

This cell is guarded and does not run unless `RUN_MARKER_ANALYSIS` is changed to `True`. It uses the public downstream analyst without retraining the model.

In [ ]:
from Main_code.get_driver import Analyst

RUN_MARKER_ANALYSIS = False

if RUN_MARKER_ANALYSIS:
    MARKER_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    analyst = Analyst(
        data_path=STAGEDATA_DIR,
        iteration=FINAL_ITERATION,
        target_dir=MARKER_RESULTS_DIR,
    )
    analysed_adata = analyst.start_analyse(
        progressionmarker_background_sampling=1000,
    )
    print(analysed_adata)

## 9. Generate integrated candidate driver genes

The five-dimensional driver-gene analysis combines gene weights, iDREM TF evidence, dynamic markers, network topology, and HC markers. The prior network is optional; omit it if it is unavailable.

In [ ]:
from Candidate_Regulator_Screening.driver_gene_identification import main as run_driver_gene_analysis

PRIOR_NETWORK_PATH = TEMP_PATH / "NicheNet_human.csv"
RUN_DRIVER_ANALYSIS = False

if RUN_DRIVER_ANALYSIS:
    DRIVER_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    driver_genes = run_driver_gene_analysis(
        temp_path=str(TEMP_PATH),
        final_iteration=FINAL_ITERATION,
        prior_net_path=str(PRIOR_NETWORK_PATH) if PRIOR_NETWORK_PATH.is_file() else None,
        output_dir=str(DRIVER_RESULTS_DIR),
        species="human",
    )
    display(driver_genes.head(30))

## 10. Generate cell-type-specific driver genes

The current project script is configured for a two-state trajectory and maps clusters to cell types before calculating per-cell-type driver genes. Change the script or the stage count if your experiment has a different design.

In [ ]:
from Candidate_Regulator_Screening import per_celltype_driver_genes

RUN_CELLTYPE_DRIVER_ANALYSIS = False

if RUN_CELLTYPE_DRIVER_ANALYSIS:
    if len(stage_files) != 2:
        raise ValueError(
            "The current per-celltype script expects exactly two stages. "
            "Adapt it before running a different number of stages."
        )
    CELLTYPE_DRIVER_DIR.mkdir(parents=True, exist_ok=True)
    celltype_driver_results = per_celltype_driver_genes.main(
        temp_path=str(TEMP_PATH),
        final_iteration=FINAL_ITERATION,
        output_dir=str(CELLTYPE_DRIVER_DIR),
    )

## 11. Classify a transcriptional cascade

The cascade module expects four cell-type driver-gene tables in `DRIVER_RESULTS_DIR`: `HSPC`, `GMP`, `Monocyte`, and `Neutrophil`. It can be called as a Python module after overriding its input and output directories.

In [ ]:
import cascade_classification.cascade_classification as cascade

RUN_CASCADE_ANALYSIS = False

if RUN_CASCADE_ANALYSIS:
    required_celltype_tables = [
        DRIVER_RESULTS_DIR / f"{cell_type}_driver_genes.csv"
        for cell_type in ["HSPC", "GMP", "Monocyte", "Neutrophil"]
    ]
    missing_tables = [path for path in required_celltype_tables if not path.is_file()]
    if missing_tables:
        raise FileNotFoundError(f"Missing cell-type driver tables: {missing_tables}")

    CASCADE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    cascade.BASE_DIR = str(DRIVER_RESULTS_DIR)
    cascade.OUT_DIR = str(CASCADE_RESULTS_DIR)
    cascade.main()

## 12. Review and save downstream outputs

All custom tables from this notebook are written below `results/downstream_tutorial/`. Core CauNagi outputs remain unchanged.

In [ ]:
output_files = (
    sorted(DOWNSTREAM_RESULTS_DIR.rglob("*"))
    if DOWNSTREAM_RESULTS_DIR.is_dir()
    else []
)
display(
    pd.DataFrame(
        {
            "file": [str(path.relative_to(PROJECT_ROOT)) for path in output_files if path.is_file()],
            "size_mb": [
                round(path.stat().st_size / 1024**2, 3)
                for path in output_files
                if path.is_file()
            ],
        }
    )
)